# EVE 310 - Lab 11: Comparing regression models

**Module 2 | 11/12/2026**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThyanRevolter/eve310-fall-2026/blob/main/labs/lab11-dairy-farm-challenge/notebooks/lab11-tutorial.ipynb)

## Learning objectives

1. Compute least-squares betas with the normal equation in NumPy; show `LinearRegression` agrees
2. Score a model on testing rows with MSE, MAE and R²: what each measures, in which units
3. Fit a random forest and spot overfitting from its training vs testing error
4. Build a column from existing ones; judge whether a simple model with the right column beats a more complex one

## Before you start

1. Click **Copy to Drive** (top of this window) and work in the copy.
2. Run the setup cell below first.
3. **Type along** with the instructor: five cells have `--` blanks. Type what the text above each says, then run the cell; **Show the finished cell** below it has the answer. **Run all** fails until every `--` is filled. Other cells: **Run as is**.
4. **Predict first** (three cells): decide what the cell will print before running it (in lab, on Menti).
5. Nothing to submit here. You download the **activity** notebook as `.ipynb` and upload it to Gradescope (optional, for feedback).

<details><summary>Why?</summary>

Colab throws away anything you did not copy when the runtime ends. The setup cell downloads the two data files into `DATA_DIR` and creates `FIGURES_DIR`. Each typed cell has one or two blanks. The finished cell under it is there if you get stuck or are working on your own. A cell with a `--` still in it cannot run. A wrong guess on a **Predict first** line is useful: it tells you what to look at in the output.

</details>

### Tutorial

A city utility needs tomorrow's peak electricity demand to decide which power plants to run; the best single clue is the weather. `power_demand.csv`: a year of a Texas city (synthetic), each day's average temperature and peak demand in megawatts (MW). Demand rises on hot days (air conditioning) and on cold days (electric heating).

**Which model predicts best on days it has not seen, and can you say why?** Three models, same 255 training days, same 110 testing days:

- **straight line**: normal equation in NumPy, then `LinearRegression`
- **random forest**: no equation, no betas
- **curve**: the line plus one column, the temperature squared

**The plan.** Every lab's six steps, named in the section titles:

| Step | What it means | In this tutorial | In the activity |
| --- | --- | --- | --- |
| **Load** | read the file into a dataframe | section 1 | the data cell |
| **Inspect** | look at the rows and a plot before any model | section 1 | the data cell |
| **Prepare** | choose `x` and `y`, split, build any new column | sections 1 and 5 | the data cell, Question 3 |
| **Fit** | create the model, then `fit` it on the training rows | sections 2, 4 and 5 | Questions 1 and 3 |
| **Check** | score the model on the testing rows and compare | sections 3 to 5 | Questions 2 and 3 |
| **Interpret** | read the results in the units of the problem | section 6 | the written answers |

Stuck on a blank cell? Find its step and ask what that step took in and gave out here.

## 0. Setup

In [ ]:
# EVE 310 setup - run this cell first, every time you open this notebook.
# It downloads this lab's data files into a "data" folder in your Colab session.
import pathlib
import urllib.request

LAB = "lab11-dairy-farm-challenge"
DATA_FILES = ["power_demand.csv", "Dairy-Farm-Dataset.csv"]

DATA_DIR = pathlib.Path("data")
FIGURES_DIR = pathlib.Path("figures")
DATA_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

BASE_URL = f"https://raw.githubusercontent.com/ThyanRevolter/eve310-fall-2026/main/labs/{LAB}/data"
for name in DATA_FILES:
    if not (DATA_DIR / name).exists():
        urllib.request.urlretrieve(f"{BASE_URL}/{name}", DATA_DIR / name)

print(f"Ready. {len(DATA_FILES)} data file(s) in {DATA_DIR.resolve()}")

**Run as is:** the imports. New: `RandomForestRegressor`, `mean_squared_error`, `mean_absolute_error`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

## 1. Load, inspect, prepare: a year of temperature and demand

**Run as is.** **Load**, **inspect** (`describe()`, a scatter plot) and **prepare** `x`, `y` and one 70/30 split for every model, so their scores can be compared.

<details><summary>Why?</summary>

`describe()` gives the range of each column, and the scatter plot shows the shape every model will have to follow. `x` is `demand_df[['temp_f']]`: two pairs of brackets give a one-column dataframe, one row per day and one column per input, the shape scikit-learn wants. `train_test_split` keeps 70% of the days for fitting and holds 30% back for testing, as in Lab 5.

</details>

In [ ]:
# Load: one row per day
demand_df = pd.read_csv(DATA_DIR / 'power_demand.csv')
print(len(demand_df), 'days')
print(demand_df.describe().round(1))

# Inspect: demand against temperature
fig, ax = plt.subplots()
ax.scatter(demand_df['temp_f'], demand_df['demand_mw'], s=12, color='k')
ax.set_xlabel('average temperature (°F)')
ax.set_ylabel('peak demand (MW)')
ax.set_title('365 days')

# Prepare: the input, the column to predict, and the 70/30 split
x = demand_df[['temp_f']]
y = demand_df['demand_mw']
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=0)
print(len(x_train), 'training days,', len(x_test), 'testing days')

Demand bottoms out near 1,400 MW on mild days (upper 50s °F) and climbs on both sides, far more steeply on the hot side: a curve no straight line can follow.

## 2. Fit: the normal equation, then `LinearRegression`

The straight line is demand = β0 + β1 × temperature. Lecture's **normal equation**, β = (XᵀX)⁻¹Xᵀy, gives the betas with the smallest sum of squared errors (SSE) on the training days. Each symbol is one variable in the cell:

| Lecture | Code | Shape here |
| --- | --- | --- |
| X, the design matrix | `X1` | 255 × 2 |
| XᵀX | `xtx` | 2 × 2 |
| (XᵀX)⁻¹ | `inv_xtx` | 2 × 2 |
| Xᵀy | `xty` | 2 |
| β | `beta` | 2 |

`X1`: a column of ones (β0 multiplies 1 on every row: the intercept), then the temperatures. `np.ones(len(x_train))`: one 1 per training day. `np.c_[a, b]`: columns side by side, in square brackets.

**Type it:** `X1 = np.c_[np.ones(len(x_train)), x_train]`, then `beta = np.dot(inv_xtx, xty)`.

In [ ]:
# The design matrix: a column of ones for the intercept, then the temperatures
--
print(X1.shape, '= (training days, columns)')

# The normal equation, beta = (X^T X)^(-1) X^T y, in the lecture notebook's four steps
xtx = np.dot(X1.T, X1)          # X-transpose times X
inv_xtx = np.linalg.inv(xtx)    # its inverse
xty = np.dot(X1.T, y_train)     # X-transpose times y
--
print('xtx =', np.round(xtx).astype(int).tolist())
print('xty =', np.round(xty).astype(int).tolist())
print('beta from the normal equation:', np.round(beta, 4))

<details><summary>Show the finished cell</summary>

```python
# The design matrix: a column of ones for the intercept, then the temperatures
X1 = np.c_[np.ones(len(x_train)), x_train]
print(X1.shape, '= (training days, columns)')

# The normal equation, beta = (X^T X)^(-1) X^T y, in the lecture notebook's four steps
xtx = np.dot(X1.T, X1)          # X-transpose times X
inv_xtx = np.linalg.inv(xtx)    # its inverse
xty = np.dot(X1.T, y_train)     # X-transpose times y
beta = np.dot(inv_xtx, xty)
print('xtx =', np.round(xtx).astype(int).tolist())
print('xty =', np.round(xty).astype(int).tolist())
print('beta from the normal equation:', np.round(beta, 4))
```

</details>

`xtx` holds four sums over the 255 training days: the number of days, the sum of the temperatures (twice) and the sum of the squared temperatures. The normal equation turns 255 days into two equations in two unknowns; `beta` solves them: β0, the intercept in MW, then β1, the slope in MW per °F.

<details><summary>More: xty</summary>

`xty` holds the sum of the demands and the sum of temperature × demand. The betas come out in the order of the columns of `X1`.

</details>

Next: the same line with `LinearRegression` (Lab 5's two lines), then a test of the formula: the training SSE of lines tilted 1 MW per °F steeper and flatter.

**Predict first:** compared with the SSE of `beta`, will the steeper line's SSE be larger, smaller, or the same? And the flatter line's?

**Type it:** the SSE of `beta`, `sse_best = np.sum((y_train - np.dot(X1, beta)) ** 2)`.

In [ ]:
lm = LinearRegression()
lm.fit(x_train, y_train)
print('intercept_ and coef_:', round(lm.intercept_, 4), np.round(lm.coef_, 4))

# The SSE on the training days: subtract the line's predictions, square, add up
--
# The same sum for two lines tilted 1 MW per °F away from beta
sse_steeper = np.sum((y_train - np.dot(X1, beta + np.array([0, 1]))) ** 2)
sse_flatter = np.sum((y_train - np.dot(X1, beta - np.array([0, 1]))) ** 2)
print('SSE of beta:               ', round(sse_best))
print('SSE, slope 1 MW/°F steeper:', round(sse_steeper))
print('SSE, slope 1 MW/°F flatter:', round(sse_flatter))
# X1 transposed times the residuals of beta: the normal equation says this is zero
print('X1.T times the residuals:', np.round(np.dot(X1.T, y_train - np.dot(X1, beta)), 3) + 0)

<details><summary>Show the finished cell</summary>

```python
lm = LinearRegression()
lm.fit(x_train, y_train)
print('intercept_ and coef_:', round(lm.intercept_, 4), np.round(lm.coef_, 4))

# The SSE on the training days: subtract the line's predictions, square, add up
sse_best = np.sum((y_train - np.dot(X1, beta)) ** 2)
# The same sum for two lines tilted 1 MW per °F away from beta
sse_steeper = np.sum((y_train - np.dot(X1, beta + np.array([0, 1]))) ** 2)
sse_flatter = np.sum((y_train - np.dot(X1, beta - np.array([0, 1]))) ** 2)
print('SSE of beta:               ', round(sse_best))
print('SSE, slope 1 MW/°F steeper:', round(sse_steeper))
print('SSE, slope 1 MW/°F flatter:', round(sse_flatter))
# X1 transposed times the residuals of beta: the normal equation says this is zero
print('X1.T times the residuals:', np.round(np.dot(X1.T, y_train - np.dot(X1, beta)), 3) + 0)
```

</details>

`intercept_` and `coef_` are `beta`'s two numbers: `fit` **is** the normal equation. Both tilted lines raise the SSE by the same 1,256,778 MW²: `beta` sits at the bottom of a bowl; every other line has a larger SSE. That is what "least squares" means.

The last line says why: Xᵀ(y − Xβ) = 0. For the column of ones, that means the residuals add up to zero.

<details><summary>Why?</summary>

XᵀXβ = Xᵀy can be rewritten as Xᵀ(y − Xβ) = 0: multiply the residuals by each column of `X1` and add up, and you get zero. For the temperature column, that means no straight-line trend with temperature is left in the residuals (a curved one can be, as section 3 shows). Tilt the line and that balance breaks, and the SSE goes up. scikit-learn solves the same equation without forming the inverse, a more careful method, so the digits agree as far as they are printed.

</details>

## 3. Check: MSE, MAE and R² on the testing days

One **error** per testing day, `y_test - y_pred`, in MW. Three summaries:

- **MSE**, mean squared error: square each error, then average. In MW². Punishes a few large misses far more than many small ones.
- **MAE**, mean absolute error: drop each error's sign (`np.abs`), then average. In MW: "off by this much on a typical day".
- **R²** (Lab 5): the share of the day-to-day variation in `y` the model explains. No units.

Smaller MSE and MAE are better; larger R² is better.

<details><summary>More: the MSE</summary>

The MSE is the SSE divided by the number of days: the quantity the normal equation just made as small as possible on the training days. Its unit, MW², is hard to read; the MAE, in MW, reads directly.

</details>

**Type it:** `mae = np.mean(np.abs(y_test - y_pred))`.

In [ ]:
# One prediction per testing day
y_pred = lm.predict(x_test)

# Mean squared error: the SSE of the testing days divided by the number of days
mse = np.mean((y_test - y_pred) ** 2)
# Mean absolute error: the errors with their signs dropped, averaged
--
r2 = r2_score(y_test, y_pred)
print('MSE:', round(mse, 1), 'MW squared')
print('MAE:', round(mae, 1), 'MW')
print('R2: ', round(r2, 3))
# scikit-learn's functions give the same two numbers
print('scikit-learn:', round(mean_squared_error(y_test, y_pred), 1),
      round(mean_absolute_error(y_test, y_pred), 1))

<details><summary>Show the finished cell</summary>

```python
# One prediction per testing day
y_pred = lm.predict(x_test)

# Mean squared error: the SSE of the testing days divided by the number of days
mse = np.mean((y_test - y_pred) ** 2)
# Mean absolute error: the errors with their signs dropped, averaged
mae = np.mean(np.abs(y_test - y_pred))
r2 = r2_score(y_test, y_pred)
print('MSE:', round(mse, 1), 'MW squared')
print('MAE:', round(mae, 1), 'MW')
print('R2: ', round(r2, 3))
# scikit-learn's functions give the same two numbers
print('scikit-learn:', round(mean_squared_error(y_test, y_pred), 1),
      round(mean_absolute_error(y_test, y_pred), 1))
```

</details>

**Run as is:** the line over the testing days.

<details><summary>More: grid</summary>

`grid` is a dataframe of temperatures from 44 to 94 °F with the same column name as `x`, so every model can draw its predictions as a line.

</details>

In [ ]:
# Temperatures from 44 to 94 °F, half a degree apart, to draw each model as a line
grid = pd.DataFrame({'temp_f': np.arange(44, 94.5, 0.5)})
fig, ax = plt.subplots()
ax.scatter(x_test['temp_f'], y_test, s=12, color='k', label='testing days')
ax.plot(grid['temp_f'], lm.predict(grid), color='tab:blue', label='straight line')
ax.set_xlabel('average temperature (°F)')
ax.set_ylabel('peak demand (MW)')
ax.legend()

The line is off by 143 MW on a typical testing day, and its errors are not random: the curve bends away from it at both ends. An MSE of about 29,850 MW² means little alone; scores are for comparing models on the same testing days.

<details><summary>More: the errors</summary>

The line predicts too little on cold days and on the hottest days, and too much in the 60s and 70s. Sections 4 and 5 compare it with two other models on the same testing days.

</details>

## 4. Fit and check a different model: a random forest

A **decision tree** asks yes/no questions learned from the training rows: *is the temperature above 78.85 °F? If so, is it above 86.6 °F?* Each chain of answers ends in a **leaf**; the tree predicts the average demand of that leaf's training days. A **random forest** averages 100 such trees; `random_state=0` fixes the trees' random resamples, so everyone's forest is the same.

<details><summary>More: the forest</summary>

Each tree is grown from a random resample of the training days and asks many more questions than two. The steps are the ones you know: create the model, `fit` it on the training days, `predict`. In Lab 10, `random_state` fixed the random starts the same way.

</details>

**Predict first:** a forest can grow a leaf for almost every training day. Will its MAE on its own training days be about the same as on the testing days, or much smaller? And the line's?

**Type it:** `rf = RandomForestRegressor(random_state=0)`, then `rf.fit(x_train, y_train)`.

In [ ]:
# A random forest: 100 trees, each grown on a random resample of the training days
--
--

# Each model's MAE on the days it was fitted on, and on the testing days
y_fit_rf = rf.predict(x_train)
y_pred_rf = rf.predict(x_test)
print('line:   MAE training days', round(mean_absolute_error(y_train, lm.predict(x_train)), 1),
      'MW, testing days', round(mae, 1), 'MW')
print('forest: MAE training days', round(mean_absolute_error(y_train, y_fit_rf), 1),
      'MW, testing days', round(mean_absolute_error(y_test, y_pred_rf), 1), 'MW')

<details><summary>Show the finished cell</summary>

```python
# A random forest: 100 trees, each grown on a random resample of the training days
rf = RandomForestRegressor(random_state=0)
rf.fit(x_train, y_train)

# Each model's MAE on the days it was fitted on, and on the testing days
y_fit_rf = rf.predict(x_train)
y_pred_rf = rf.predict(x_test)
print('line:   MAE training days', round(mean_absolute_error(y_train, lm.predict(x_train)), 1),
      'MW, testing days', round(mae, 1), 'MW')
print('forest: MAE training days', round(mean_absolute_error(y_train, y_fit_rf), 1),
      'MW, testing days', round(mean_absolute_error(y_test, y_pred_rf), 1), 'MW')
```

</details>

**Run as is:** the forest over the testing days, next to the line.

In [ ]:
fig, ax = plt.subplots()
ax.scatter(x_test['temp_f'], y_test, s=12, color='k', label='testing days')
ax.plot(grid['temp_f'], lm.predict(grid), color='tab:blue', label='straight line')
ax.plot(grid['temp_f'], rf.predict(grid), color='tab:orange', label='random forest')
ax.set_xlabel('average temperature (°F)')
ax.set_ylabel('peak demand (MW)')
ax.legend()

The forest follows the curve in small steps and halves the line's testing MAE: 72 MW against 143. Compare each model with itself, training against testing days:

- line: 141 against 143 MW, about the same
- forest: 35 against 72 MW: it has partly memorized its training days, noise included

That is **overfitting**: a model much better on the rows it was fitted on than on new rows. It is why models are judged on testing rows only.

Compare MAEs, not R², for this: R² also depends on how spread out each set of days is, and these testing days happen to be less spread out, lowering every model's testing R².

<details><summary>More: the gap</summary>

The forest still wins on the testing days here; the gap between its training and testing MAE says how much of its training score to believe.

</details>

## 5. Your turn (prepare, fit, check): one more column, and the line becomes a curve

New column: `temp_sq`, the temperature squared. Still `LinearRegression`, still the normal equation, now with three betas: demand = β0 + β1 × temperature + β2 × temperature². Linear in the betas, curved in the temperature.

`x_train_sq` and `x_test_sq` are **copies** (`.copy()`): the line's and the forest's dataframes stay unchanged. Fill each `--` with that copy's own temperature column, squared (`** 2`, as in section 2). The testing copy needs the column too: a model fitted on two columns must be given the same two to predict.

<details><summary>More: new columns</summary>

Lab 6 built new columns from old ones: `Temp_x_Med` was a product of two columns.

</details>

In [ ]:
# Copies of x_train and x_test with a second column, the temperature squared
x_train_sq = x_train.copy()
x_train_sq['temp_sq'] = --
x_test_sq = x_test.copy()
x_test_sq['temp_sq'] = --

lm_sq = LinearRegression()
lm_sq.fit(x_train_sq, y_train)
y_pred_sq = lm_sq.predict(x_test_sq)
print('betas:', round(lm_sq.intercept_, 2), np.round(lm_sq.coef_, 5))
print('curve:  MAE training days', round(mean_absolute_error(y_train, lm_sq.predict(x_train_sq)), 1),
      'MW, testing days', round(mean_absolute_error(y_test, y_pred_sq), 1), 'MW')

<details><summary>Show the finished cell</summary>

```python
# Copies of x_train and x_test with a second column, the temperature squared
x_train_sq = x_train.copy()
x_train_sq['temp_sq'] = x_train['temp_f'] ** 2
x_test_sq = x_test.copy()
x_test_sq['temp_sq'] = x_test['temp_f'] ** 2

lm_sq = LinearRegression()
lm_sq.fit(x_train_sq, y_train)
y_pred_sq = lm_sq.predict(x_test_sq)
print('betas:', round(lm_sq.intercept_, 2), np.round(lm_sq.coef_, 5))
print('curve:  MAE training days', round(mean_absolute_error(y_train, lm_sq.predict(x_train_sq)), 1),
      'MW, testing days', round(mean_absolute_error(y_test, y_pred_sq), 1), 'MW')
```

</details>

**Run as is:** the three models over the testing days. The curve, `lm_sq`, needs `grid` with the squared column too.

In [ ]:
grid_sq = grid.copy()
grid_sq['temp_sq'] = grid['temp_f'] ** 2
fig, ax = plt.subplots()
ax.scatter(x_test['temp_f'], y_test, s=12, color='k', label='testing days')
ax.plot(grid['temp_f'], lm.predict(grid), color='tab:blue', label='straight line')
ax.plot(grid['temp_f'], rf.predict(grid), color='tab:orange', label='random forest')
ax.plot(grid['temp_f'], lm_sq.predict(grid_sq), color='tab:green', label='curve')
ax.set_xlabel('average temperature (°F)')
ax.set_ylabel('peak demand (MW)')
ax.legend()

Testing MAE 58 MW: better than the forest's 72, from a model you can write in one line. No overfitting: 60 MW on training days, 58 on testing. The forest had to find the data's curve step by step; the squared column hands the shape to a simpler model.

## 6. Interpret: the scores side by side, and two new days

**Run as is.** The scores side by side, then two new days: a mild 75 °F day and a 105 °F heat wave, hotter than any day in the file.

<details><summary>More: the new days</summary>

`new_days` is a dataframe with the same column name as `x`, and `new_days_sq` adds the squared column, because `lm_sq` was fitted on both.

</details>

**Predict first:** for the 105 °F day, which model will predict the highest demand, and which the lowest?

In [ ]:
scores_df = pd.DataFrame({
    'model': ['straight line', 'random forest', 'curve'],
    'MAE training (MW)': [mean_absolute_error(y_train, lm.predict(x_train)), mean_absolute_error(y_train, y_fit_rf),
                          mean_absolute_error(y_train, lm_sq.predict(x_train_sq))],
    'MAE testing (MW)': [mae, mean_absolute_error(y_test, y_pred_rf), mean_absolute_error(y_test, y_pred_sq)],
    'R2 testing': [r2, r2_score(y_test, y_pred_rf), r2_score(y_test, y_pred_sq)],
})
print(scores_df.round(3))

# Two new days: a mild one, and a heat wave hotter than any day in the data
new_days = pd.DataFrame({'temp_f': [75.0, 105.0]})
new_days_sq = new_days.copy()
new_days_sq['temp_sq'] = new_days['temp_f'] ** 2
print('straight line:', np.round(lm.predict(new_days)))
print('random forest:', np.round(rf.predict(new_days)))
print('curve:        ', np.round(lm_sq.predict(new_days_sq)))

At 75 °F the three models agree within about 150 MW; at 105 °F they are far apart: line about 2,770 MW, curve about 3,950 MW, forest 2,876 MW.

- **Forest**: averages of training days, so never above the highest training demand (2,916 MW). Every day hotter than the hottest training day (93.6 °F) gets the same 2,876 MW.
- **Line and curve**: keep their own shapes past the data, a guess too; none of the three has seen a 105 °F day.

Every model is least trustworthy outside the range it was fitted on: exactly when a utility most needs the forecast.

**Three things to remember**

1. **`fit` is the normal equation.** A squared column, or a product of two columns, is one more column of `X`: still linear regression.
2. **Judge a model on the testing rows, in units someone can act on.** MAE: units of `y`. MSE: squared units, punishes big misses. R²: no units. Much smaller training error than testing error: overfitting.
3. **The right column can beat a more complex model**, and it keeps betas you can explain. No model is safe outside its training range; a forest never predicts above the largest or below the smallest training `y`.

<details><summary><b>Common mistakes</b>: open this when a cell fails</summary>

| What you see | What went wrong |
| --- | --- |
| `TypeError: 'CClass' object is not callable` | `np.c_(...)` with parentheses. `np.c_` takes square brackets: `np.c_[a, b]` |
| `TypeError: Cannot interpret ... as a data type` | A bracket in the wrong place, `np.ones(len(x_train), x_train)`. Close `np.ones(len(x_train))` before the comma |
| `ValueError: all the input array dimensions except for the concatenation axis must match exactly` | The ones and `x_train` have different lengths, for example `np.ones(len(x))`: 365 ones against 255 training days |
| `beta` prints huge numbers, such as `[5.72316292e+11 4.08692237e+13]` | `np.dot(xtx, xty)`: the inverse is missing. The first factor is `inv_xtx`, (XᵀX)⁻¹ |
| `beta` has two numbers, `lm.coef_` only one | Not a mistake: `beta[0]` is `lm.intercept_` and `beta[1]` is `lm.coef_[0]`. `coef_` holds the slopes only |
| `ValueError: operands could not be broadcast together with shapes (255,) (110,)` | Training and testing arrays in one line, such as `y_train - y_pred`: 255 training demands against 110 predictions. The testing errors are `y_test - y_pred` |
| The MAE is near 0, or negative | `np.abs` is missing: `np.mean(y_test - y_pred)` lets positive and negative errors cancel |
| The forest's numbers differ from your neighbor's | `random_state=0` is missing, so your forest grew from other random resamples |
| `TypeError: Cannot perform 'xor' with a dtyped [float64] array and scalar of type [bool]` | `^ 2` is not a power in Python. Square with `** 2` |
| `ValueError: Input X contains NaN` | A copy's column was built from the other set's rows, such as `x_test_sq['temp_sq'] = x_train['temp_f'] ** 2`. pandas lines rows up by their index, and no testing day is a training day, so every value is missing (NaN). Build each copy's column from its own rows |

</details>

## 7. Why this matters

- **A forecast is a decision.** The costliest misses come on the hottest and coldest days, where the line is worst and the forest goes flat past its training range.
- **Betas explain, forests predict.** Betas say *how much* the prediction moves with each column; in the activity, how much each farm practice changes the predicted carbon footprint of milk. A forest only says *what*: sometimes all you need, sometimes not enough.

Now open the **activity** notebook.

## Wrap-up

Keep this notebook open for the activity: a linear model and a forest on 7,000 dairy farms (scored on the other 3,000), plus a column built from the data's units, not a plot. Its Question 3 gives the steps as comments; the code is yours, so keep the step table above in view.

At home: drag a line through this tutorial's training days in the [least-squares explorer](https://thyanrevolter.github.io/eve310-fall-2026/labs/lab11-least-squares-explorer.html) and watch the residuals and the SSE.